In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Regression Models
from sklearn.linear_model import Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Model Tuning
import optuna

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [3]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [4]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [5]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [6]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
TARGET = kolom_info['target']

In [8]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [9]:
print('=== VERIFIKASI LOAD DATA ===')
print(f'X_train : {X_train.shape}')
print(f'X_val : {X_val.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train : {y_train_arr.shape}')
print(f'y_val : {y_val_arr.shape}')
print(f'y_test : {y_test_arr.shape}')
print(f'\nTotal fitur: {X_train.shape[1]}')
print(f'Target : {TARGET}')
print(f'\nNull check:')
print(f' X_train: {X_train.isnull().sum().sum()}')
print(f' X_val : {X_val.isnull().sum().sum()}')
print(f' X_test : {X_test.isnull().sum().sum()}')
print(f'\nRange target:')
print(f' Train: {y_train_arr.min():.2f} - {y_train_arr.max():.2f} Qu/Ha')
print(f' Val : {y_val_arr.min():.2f} - {y_val_arr.max():.2f} Qu/Ha')
print(f' Test : {y_test_arr.min():.2f} - {y_test_arr.max():.2f} Qu/Ha')
print(f'\nArtifak loaded:')
print(f' minmax_scaler : OK')
print(f' kolom_info : {len(kolom_info)} entri')
print(f' winsor_bounds : {len(winsor)} kolom')
print(f'\nFolder output:')
print(f' Model : {MODEL_FOLDER}')
print(f' Gambar : {EDA_FOLDER}')
print('\nData siap. Lanjut ke modeling.')

=== VERIFIKASI LOAD DATA ===
X_train : (2251, 85)
X_val : (450, 85)
X_test : (893, 85)
y_train : (2251,)
y_val : (450,)
y_test : (893,)

Total fitur: 85
Target : produktivitas_kuha

Null check:
 X_train: 0
 X_val : 0
 X_test : 0

Range target:
 Train: 12.32 - 75.54 Qu/Ha
 Val : 18.63 - 71.66 Qu/Ha
 Test : 12.54 - 75.32 Qu/Ha

Artifak loaded:
 minmax_scaler : OK
 kolom_info : 12 entri
 winsor_bounds : 59 kolom

Folder output:
 Model : dataset/model_outputs
 Gambar : dataset/eda_outputs

Data siap. Lanjut ke modeling.


In [10]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [11]:
print("Mulai tuning Stacking Ensemble (Bayesian - Optuna)")

def objective(trial):
    # suggest base model parameters
    lasso_alpha = trial.suggest_float("lasso_alpha", 0.001, 0.1, log=True)
    svr_C = trial.suggest_float("svr_C", 1.0, 20.0)
    svr_epsilon = trial.suggest_float("svr_epsilon", 0.05, 0.2)
    rf_n_est = trial.suggest_int("rf_n_estimators", 100, 300, step=50)
    rf_depth = trial.suggest_categorical("rf_max_depth", [10, 15, 20])
    gb_n_est = trial.suggest_int("gb_n_estimators", 100, 300, step=50)
    gb_lr = trial.suggest_float("gb_learning_rate", 0.03, 0.1)
    meta_alpha = trial.suggest_float("meta_alpha", 0.0005, 0.01, log=True)
    
    # build base models
    base_models = [
        ("lasso", Lasso(alpha=lasso_alpha, max_iter=10000)),
        ("svr", SVR(C=svr_C, epsilon=svr_epsilon, kernel="rbf")),
        ("rf", RandomForestRegressor(
            n_estimators=rf_n_est,
            max_depth=rf_depth,
            random_state=42,
            n_jobs=-1
        )),
        ("gb", GradientBoostingRegressor(
            n_estimators=gb_n_est,
            learning_rate=gb_lr,
            random_state=42
        ))
    ]
    
    # build stacking with meta model
    model = StackingRegressor(
        estimators=base_models,
        final_estimator=Lasso(alpha=meta_alpha, max_iter=10000)
    )
    
    model.fit(X_train, y_train_arr)
    val_pred = model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val_arr, val_pred))
    return val_rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_params
best_rmse = study.best_value

# rebuild model dengan best params
base_models = [
    ("lasso", Lasso(alpha=best_params["lasso_alpha"], max_iter=10000)),
    ("svr", SVR(C=best_params["svr_C"], epsilon=best_params["svr_epsilon"], kernel="rbf")),
    ("rf", RandomForestRegressor(
        n_estimators=best_params["rf_n_estimators"],
        max_depth=best_params["rf_max_depth"],
        random_state=42,
        n_jobs=-1
    )),
    ("gb", GradientBoostingRegressor(
        n_estimators=best_params["gb_n_estimators"],
        learning_rate=best_params["gb_learning_rate"],
        random_state=42
    ))
]

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Lasso(alpha=best_params["meta_alpha"], max_iter=10000)
)
stacking_model.fit(X_train, y_train_arr)

print("Tuning selesai")
print("Best Validation RMSE:", best_rmse)
print("Best params:")
print(best_params)

[I 2026-06-02 19:26:02,357] A new study created in memory with name: no-name-245fc222-9b77-4e94-adeb-802ea9e1cf7d


Mulai tuning Stacking Ensemble (Bayesian - Optuna)


Best trial: 0. Best value: 5.47128:   5%|▌         | 1/20 [00:31<10:00, 31.62s/it]

[I 2026-06-02 19:26:33,977] Trial 0 finished with value: 5.4712765352239 and parameters: {'lasso_alpha': 0.0028581035389748217, 'svr_C': 14.514455926875666, 'svr_epsilon': 0.15895068811344013, 'rf_n_estimators': 150, 'rf_max_depth': 15, 'gb_n_estimators': 300, 'gb_learning_rate': 0.06336304354850442, 'meta_alpha': 0.005449767846677967}. Best is trial 0 with value: 5.4712765352239.


Best trial: 0. Best value: 5.47128:  10%|█         | 2/20 [01:05<09:49, 32.75s/it]

...

[I 2026-06-02 19:33:28,384] Trial 17 finished with value: 5.529533543908195 and parameters: {'lasso_alpha': 0.03890475316350824, 'svr_C': 16.14163137123407, 'svr_epsilon': 0.18509026502789244, 'rf_n_estimators': 300, 'rf_max_depth': 10, 'gb_n_estimators': 300, 'gb_learning_rate': 0.04581656545276064, 'meta_alpha': 0.008740102653796175}. Best is trial 11 with value: 5.420060795420202.


Best trial: 18. Best value: 5.41602:  95%|█████████▌| 19/20 [07:48<00:26, 26.39s/it]

[I 2026-06-02 19:33:50,422] Trial 18 finished with value: 5.416023361380402 and parameters: {'lasso_alpha': 0.012658121977743868, 'svr_C': 19.905540723070224, 'svr_epsilon': 0.16425777275215125, 'rf_n_estimators': 100, 'rf_max_depth': 15, 'gb_n_estimators': 200, 'gb_learning_rate': 0.07477398404743972, 'meta_alpha': 0.005652116283392699}. Best is trial 18 with value: 5.416023361380402.


Best trial: 18. Best value: 5.41602: 100%|██████████| 20/20 [08:09<00:00, 24.49s/it]


[I 2026-06-02 19:34:12,135] Trial 19 finished with value: 5.460964891830136 and parameters: {'lasso_alpha': 0.01172627273071869, 'svr_C': 19.828424667686125, 'svr_epsilon': 0.054899644113333965, 'rf_n_estimators': 100, 'rf_max_depth': 15, 'gb_n_estimators': 200, 'gb_learning_rate': 0.07332183840318608, 'meta_alpha': 0.005575330079307966}. Best is trial 18 with value: 5.416023361380402.
Tuning selesai
Best Validation RMSE: 5.416023361380402
Best params:
{'lasso_alpha': 0.012658121977743868, 'svr_C': 19.905540723070224, 'svr_epsilon': 0.16425777275215125, 'rf_n_estimators': 100, 'rf_max_depth': 15, 'gb_n_estimators': 200, 'gb_learning_rate': 0.07477398404743972, 'meta_alpha': 0.005652116283392699}


In [12]:
# Prediksi
train_pred = stacking_model.predict(X_train)
val_pred = stacking_model.predict(X_val)
test_pred = stacking_model.predict(X_test)

In [13]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [14]:
# evaluasi train
train_mae, train_rmse, train_r2 = hitung_metrics(
    y_train_arr,
    train_pred
)

In [15]:
# evaluasi validation
val_mae, val_rmse, val_r2 = hitung_metrics(
    y_val_arr,
    val_pred
)

In [16]:
# evaluasi test
test_mae, test_rmse, test_r2 = hitung_metrics(
    y_test_arr,
    test_pred
)

In [17]:
print("HASIL STACKING ENSEMBLE")

print("\nTrain")
print(f"MAE  : {train_mae:.4f}")
print(f"RMSE : {train_rmse:.4f}")
print(f"R2   : {train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R2   : {val_r2:.4f}")

print("\nTest")
print(f"MAE  : {test_mae:.4f}")
print(f"RMSE : {test_rmse:.4f}")
print(f"R2   : {test_r2:.4f}")

HASIL STACKING ENSEMBLE

Train
MAE  : 3.7913
RMSE : 4.9873
R2   : 0.7841

Validation
MAE  : 4.1682
RMSE : 5.4160
R2   : 0.7229

Test
MAE  : 4.3511
RMSE : 5.6733
R2   : 0.7112


In [18]:
# Simpan Model Linear Regression
joblib.dump(
    stacking_model,
    os.path.join(BASE_PATH, 'models', 'stacking_ensemble.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [19]:
# Simpan metrics
metrics_lr = pd.DataFrame({
    "model": ["Stacking Ensemble"],
    "train_mae": [train_mae],
    "train_rmse": [train_rmse],
    "train_r2": [train_r2],
    "val_mae": [val_mae],
    "val_rmse": [val_rmse],
    "val_r2": [val_r2],
    "test_mae": [test_mae],
    "test_rmse": [test_rmse],
    "test_r2": [test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_lr.to_csv(
    os.path.join(BASE_PATH, "results", "stacking_ensemble_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [20]:
# Simpan hasil prediksi test
hasil_pred_stacking = ID_test.copy()

hasil_pred_stacking["actual"] = y_test.values
hasil_pred_stacking["prediction_stacking_ensemble"] = test_pred

hasil_pred_stacking.to_csv(
    os.path.join(BASE_PATH, "results", "stacking_ensemble_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_stacking.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  \
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82   
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53   
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15   
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77   
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82   

   prediction_stacking_ensemble  
0                     49.626734  
1                     52.433119  
2                     54.044173  
3                     56.711077  
4                     49.863737  


In [22]:
from sklearn.inspection import permutation_importance

# StackingRegressor doesn't expose feature_importances_; use permutation importance instead.
perm_res = permutation_importance(
    stacking_model,
    X_val,
    y_val_arr,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring="neg_mean_squared_error"
)

feature_importance_xgb = pd.DataFrame({
    "feature": X_train.columns,
    "importance": np.abs(perm_res.importances_mean)
})

feature_importance_xgb = feature_importance_xgb.sort_values(
    by="importance",
    ascending=False
)

feature_importance_xgb.to_csv(
    os.path.join(BASE_PATH, "results", 'stacking_permutation_feature_importance.csv'),
    index=False
)

print("Hasil Stacking Ensemble permutation feature importance berhasil disimpan.")

Hasil Stacking Ensemble permutation feature importance berhasil disimpan.
